### 1. Load the Baseline and ICSA Triple Files

In [1]:
# 1. Load the Baseline and ICSA Triple Files

from pathlib import Path
import pandas as pd

current_dir = Path.cwd().resolve()
BASE_DIR = next(
    (path for path in [current_dir, *current_dir.parents] if (path / 'data').exists()),
    None
)

if BASE_DIR is None:
    raise FileNotFoundError("Could not find the project root containing the 'data' directory.")

EMBEDDING_DIR = BASE_DIR / 'data' / 'processed' / 'embedding' / '2024_12'

TRAIN_PATH = EMBEDDING_DIR / 'nvd_threat_kg_train.csv'
VALID_PATH = EMBEDDING_DIR / 'nvd_threat_kg_valid.csv'
TEST_PATH = EMBEDDING_DIR / 'nvd_threat_kg_test.csv'
ICSA_PATH = BASE_DIR / 'data' / 'processed' / 'kg' / '2024_12' / 'icsa_threat_kg.csv'


def load_triple_csv(path):
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]

    # Case 1: already standardized
    if {'head', 'relation', 'tail'}.issubset(df.columns):
        return df[['head', 'relation', 'tail']].copy()

    # Case 2: existing NVD triple format
    if {'subject', 'predicate', 'object'}.issubset(df.columns):
        return df.rename(columns={
            'subject': 'head',
            'predicate': 'relation',
            'object': 'tail',
        })[['head', 'relation', 'tail']].copy()

    # Case 3: generic 3-column file
    if len(df.columns) == 3:
        df = df.copy()
        df.columns = ['head', 'relation', 'tail']
        return df

    raise ValueError(f'Unexpected triple columns in {path.name}: {df.columns.tolist()}')


nvd_train_df = load_triple_csv(TRAIN_PATH)
nvd_valid_df = load_triple_csv(VALID_PATH)
nvd_test_df = load_triple_csv(TEST_PATH)
icsa_df = load_triple_csv(ICSA_PATH)

print(f'[INFO] Train triples: {len(nvd_train_df)}')
print(f'[INFO] Valid triples: {len(nvd_valid_df)}')
print(f'[INFO] Test triples:  {len(nvd_test_df)}')
print(f'[INFO] ICSA triples:  {len(icsa_df)}')

print('[INFO] Train columns:', nvd_train_df.columns.tolist())
print('[INFO] ICSA columns:', icsa_df.columns.tolist())

display(nvd_train_df.head())
display(icsa_df.head())

[INFO] Train triples: 1685929
[INFO] Valid triples: 10000
[INFO] Test triples:  10000
[INFO] ICSA triples:  17071
[INFO] Train columns: ['head', 'relation', 'tail']
[INFO] ICSA columns: ['head', 'relation', 'tail']


,head,relation,tail
0,CVE-2016-0002,MatchingCWE,CWE-119
1,CVE-2016-0003,MatchingCWE,CWE-119
2,CVE-2016-0005,MatchingCWE,CWE-20
3,CVE-2016-0008,MatchingCWE,CWE-200
4,CVE-2016-0010,MatchingCWE,CWE-119


,head,relation,tail
0,ICSA-16-014-01,includesCVE,CVE-2015-3943
1,ICSA-16-014-01,includesCVE,CVE-2015-3946
2,ICSA-16-014-01,includesCVE,CVE-2015-3947
3,ICSA-16-014-01,includesCVE,CVE-2015-3948
4,ICSA-16-014-01,includesCVE,CVE-2015-6467


### 2. Build CVE Sets for Each Split

In [2]:
# 2. Build CVE Sets for Each Split

import re

CVE_PATTERN = re.compile(r'^CVE-\d{4}-\d+$', re.IGNORECASE)


def extract_cve_set(df):
    values = pd.concat([df['head'], df['tail']], ignore_index=True)
    values = values.dropna().astype(str).str.strip()
    return {
        value.upper()
        for value in values
        if CVE_PATTERN.match(value)
    }


train_cves = extract_cve_set(nvd_train_df)
valid_cves = extract_cve_set(nvd_valid_df)
test_cves = extract_cve_set(nvd_test_df)

valid_test_cves = valid_cves | test_cves

holdout_cves = valid_cves | test_cves
train_only_cves = train_cves - holdout_cves

print(f'[INFO] # Holdout CVEs (valid ∪ test): {len(holdout_cves)}')
print(f'[INFO] # Train-only CVEs: {len(train_only_cves)}')

[INFO] # Holdout CVEs (valid ∪ test): 8339
[INFO] # Train-only CVEs: 182485


In [8]:
# 3. Filter Eligible ICSA Groups for Training Augmentation

icsa_train_group_dfs = []
icsa_group_stats = []

for advisory_id, group in icsa_df.groupby('head', sort=False):
    group = group.copy()

    group_cves = {
        value.upper()
        for value in group.loc[group['relation'] == 'includesCVE', 'tail'].dropna().astype(str)
    }
    group_cpes = {
        value
        for value in group.loc[group['relation'] == 'affectsCPE', 'tail'].dropna().astype(str)
    }

    has_holdout_cve = len(group_cves & holdout_cves) > 0
    train_linked_cves = sorted(group_cves & train_only_cves)

    keep_group = (not has_holdout_cve) and (len(train_linked_cves) > 0)

    icsa_group_stats.append({
        'advisory_id': advisory_id,
        'num_group_cves': len(group_cves),
        'num_group_cpes': len(group_cpes),
        'num_train_only_linked_cves': len(train_linked_cves),
        'has_holdout_cve': has_holdout_cve,
        'keep_group': keep_group,
    })

    if keep_group:
        filtered_group = pd.concat([
            group[
                (group['relation'] == 'includesCVE') &
                (group['tail'].str.upper().isin(train_linked_cves))
            ],
            group[group['relation'] == 'affectsCPE']
        ], ignore_index=True).drop_duplicates()

        icsa_train_group_dfs.append(filtered_group)

icsa_group_stats_df = pd.DataFrame(icsa_group_stats)

if len(icsa_train_group_dfs) > 0:
    icsa_train_df = pd.concat(icsa_train_group_dfs, ignore_index=True).drop_duplicates().reset_index(drop=True)
else:
    icsa_train_df = pd.DataFrame(columns=['head', 'relation', 'tail'])

print(f'[INFO] Total ICSA groups: {len(icsa_group_stats_df)}')
print(f'[INFO] Kept ICSA groups: {icsa_group_stats_df["keep_group"].sum()}')
print(f'[INFO] Filtered ICSA triples for training: {len(icsa_train_df)}')

display(icsa_group_stats_df.head())
display(icsa_train_df.head(20))

[INFO] Total ICSA groups: 2436
[INFO] Kept ICSA groups: 2085
[INFO] Filtered ICSA triples for training: 10216


,advisory_id,num_group_cves,num_group_cpes,num_train_only_linked_cves,has_holdout_cve,keep_group
0,ICSA-16-014-01,15,1,10,False,True
1,ICSA-16-019-01,1,2,1,False,True
2,ICSA-16-021-01,1,1,1,False,True
3,ICSA-16-026-01,1,1,1,False,True
4,ICSA-16-026-02,1,8,1,False,True


,head,relation,tail
0,ICSA-16-014-01,includesCVE,CVE-2016-0851
1,ICSA-16-014-01,includesCVE,CVE-2016-0852
2,ICSA-16-014-01,includesCVE,CVE-2016-0853
3,ICSA-16-014-01,includesCVE,CVE-2016-0854
4,ICSA-16-014-01,includesCVE,CVE-2016-0855
5,ICSA-16-014-01,includesCVE,CVE-2016-0856
6,ICSA-16-014-01,includesCVE,CVE-2016-0857
7,ICSA-16-014-01,includesCVE,CVE-2016-0858
8,ICSA-16-014-01,includesCVE,CVE-2016-0859
9,ICSA-16-014-01,includesCVE,CVE-2016-0860


In [9]:
# 4. Inspect the Filtered ICSA Training Triples

icsa_train_relation_summary = (
    icsa_train_df['relation']
    .value_counts()
    .rename_axis('relation')
    .reset_index(name='count')
)

print('[INFO] Relation counts in filtered ICSA training triples')
display(icsa_train_relation_summary)

print('[INFO] Sample kept ICSA groups')
display(
    icsa_group_stats_df[icsa_group_stats_df['keep_group'] == True]
    .head(20)
)

[INFO] Relation counts in filtered ICSA training triples


,relation,count
0,affectsCPE,5419
1,includesCVE,4797


[INFO] Sample kept ICSA groups


,advisory_id,num_group_cves,num_group_cpes,num_train_only_linked_cves,has_holdout_cve,keep_group
0,ICSA-16-014-01,15,1,10,False,True
1,ICSA-16-019-01,1,2,1,False,True
2,ICSA-16-021-01,1,1,1,False,True
3,ICSA-16-026-01,1,1,1,False,True
4,ICSA-16-026-02,1,8,1,False,True
6,ICSA-16-033-02,2,0,2,False,True
7,ICSA-16-040-01,4,0,4,False,True
8,ICSA-16-040-02,2,1,2,False,True
9,ICSA-16-049-01,1,0,1,False,True
10,ICSA-16-056-01,1,1,1,False,True


In [10]:
# 5. Merge the Baseline Training Triples with the ICSA Triples

nvd_icsa_train_df = (
    pd.concat([nvd_train_df, icsa_train_df], ignore_index=True)
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f'[INFO] Baseline train triples: {len(nvd_train_df)}')
print(f'[INFO] Added ICSA train triples: {len(icsa_train_df)}')
print(f'[INFO] Augmented train triples: {len(nvd_icsa_train_df)}')
print(f'[INFO] Net increase: {len(nvd_icsa_train_df) - len(nvd_train_df)}')

display(nvd_icsa_train_df.head())

[INFO] Baseline train triples: 1685929
[INFO] Added ICSA train triples: 10216
[INFO] Augmented train triples: 1696145
[INFO] Net increase: 10216


,head,relation,tail
0,CVE-2016-0002,MatchingCWE,CWE-119
1,CVE-2016-0003,MatchingCWE,CWE-119
2,CVE-2016-0005,MatchingCWE,CWE-20
3,CVE-2016-0008,MatchingCWE,CWE-200
4,CVE-2016-0010,MatchingCWE,CWE-119


In [12]:
# 6. Save the Augmented Split Files

AUG_TRAIN_PATH = EMBEDDING_DIR / 'nvd_icsa_threat_kg_train.csv'
AUG_VALID_PATH = EMBEDDING_DIR / 'nvd_icsa_threat_kg_valid.csv'
AUG_TEST_PATH = EMBEDDING_DIR / 'nvd_icsa_threat_kg_test.csv'

ICSA_TRAIN_ONLY_PATH = EMBEDDING_DIR / 'icsa_train_filtered.csv'
ICSA_GROUP_STATS_PATH = EMBEDDING_DIR / 'icsa_group_filter_summary.csv'


def to_spo_format(df):
    return df.rename(columns={
        'head': 'subject',
        'relation': 'predicate',
        'tail': 'object'
    })[['subject', 'predicate', 'object']].copy()


to_spo_format(nvd_icsa_train_df).to_csv(AUG_TRAIN_PATH, index=False)
to_spo_format(nvd_valid_df).to_csv(AUG_VALID_PATH, index=False)
to_spo_format(nvd_test_df).to_csv(AUG_TEST_PATH, index=False)
to_spo_format(icsa_train_df).to_csv(ICSA_TRAIN_ONLY_PATH, index=False)

icsa_group_stats_df.to_csv(ICSA_GROUP_STATS_PATH, index=False)

print(f'[DONE] Saved augmented train file to: {AUG_TRAIN_PATH}')
print(f'[DONE] Saved valid file to: {AUG_VALID_PATH}')
print(f'[DONE] Saved test file to:  {AUG_TEST_PATH}')
print(f'[DONE] Saved filtered ICSA triples to: {ICSA_TRAIN_ONLY_PATH}')
print(f'[DONE] Saved ICSA group summary to: {ICSA_GROUP_STATS_PATH}')

[DONE] Saved augmented train file to: C:\Workspace\icsa-threat-kg\data\processed\embedding\2024_12\nvd_icsa_threat_kg_train.csv
[DONE] Saved valid file to: C:\Workspace\icsa-threat-kg\data\processed\embedding\2024_12\nvd_icsa_threat_kg_valid.csv
[DONE] Saved test file to:  C:\Workspace\icsa-threat-kg\data\processed\embedding\2024_12\nvd_icsa_threat_kg_test.csv
[DONE] Saved filtered ICSA triples to: C:\Workspace\icsa-threat-kg\data\processed\embedding\2024_12\icsa_train_filtered.csv
[DONE] Saved ICSA group summary to: C:\Workspace\icsa-threat-kg\data\processed\embedding\2024_12\icsa_group_filter_summary.csv


In [5]:
# 7. Load the Augmented Train/Valid/Test Triples

import pandas as pd
from pathlib import Path

AUG_TRAIN_PATH = EMBEDDING_DIR / 'nvd_icsa_threat_kg_train.csv'
AUG_VALID_PATH = EMBEDDING_DIR / 'nvd_icsa_threat_kg_valid.csv'
AUG_TEST_PATH = EMBEDDING_DIR / 'nvd_icsa_threat_kg_test.csv'

train_df = pd.read_csv(AUG_TRAIN_PATH)
valid_df = pd.read_csv(AUG_VALID_PATH)
test_df = pd.read_csv(AUG_TEST_PATH)

print(f'[INFO] Augmented train triples: {len(train_df)}')
print(f'[INFO] Valid triples: {len(valid_df)}')
print(f'[INFO] Test triples: {len(test_df)}')

display(train_df.head())

[INFO] Augmented train triples: 1696145
[INFO] Valid triples: 10000
[INFO] Test triples: 10000


,subject,predicate,object
0,CVE-2016-0002,MatchingCWE,CWE-119
1,CVE-2016-0003,MatchingCWE,CWE-119
2,CVE-2016-0005,MatchingCWE,CWE-20
3,CVE-2016-0008,MatchingCWE,CWE-200
4,CVE-2016-0010,MatchingCWE,CWE-119


In [7]:
# 8. Build TriplesFactory Objects

from pykeen.triples import TriplesFactory

train_triples = train_df[['subject', 'predicate', 'object']].astype(str).values
valid_triples = valid_df[['subject', 'predicate', 'object']].astype(str).values
test_triples = test_df[['subject', 'predicate', 'object']].astype(str).values

training_factory = TriplesFactory.from_labeled_triples(train_triples)
validation_factory = TriplesFactory.from_labeled_triples(
    valid_triples,
    entity_to_id=training_factory.entity_to_id,
    relation_to_id=training_factory.relation_to_id,
)
testing_factory = TriplesFactory.from_labeled_triples(
    test_triples,
    entity_to_id=training_factory.entity_to_id,
    relation_to_id=training_factory.relation_to_id,
)

print(training_factory)
print(validation_factory)
print(testing_factory)

TriplesFactory(num_entities=448384, num_relations=20, create_inverse_triples=False, num_triples=1696145)
TriplesFactory(num_entities=448384, num_relations=20, create_inverse_triples=False, num_triples=10000)
TriplesFactory(num_entities=448384, num_relations=20, create_inverse_triples=False, num_triples=10000)


In [8]:
# 9. Train and Evaluate the TransE Model on the ICSA-Augmented KG

import torch
from pykeen.pipeline import pipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[INFO] Using device: {device}')

result_transe = pipeline(
    training=training_factory,
    validation=validation_factory,
    testing=testing_factory,
    model='TransE',
    model_kwargs=dict(
        embedding_dim=200,
        scoring_fct_norm=1,
    ),
    training_loop='sLCWA',
    negative_sampler='basic',
    negative_sampler_kwargs=dict(
        num_negs_per_pos=20,
        corruption_scheme=('head', 'tail'),
    ),
    training_kwargs=dict(
        num_epochs=300,
        batch_size=4096,
        use_tqdm=True,
    ),
    optimizer='adam',
    optimizer_kwargs=dict(lr=1e-4),
    loss='nssa',
    regularizer='lp',
    regularizer_kwargs=dict(
        p=3,
        weight=1e-5,
    ),
    device=device,
    random_seed=1,
)

metric_results = result_transe.metric_results

try:
    mrr = metric_results.get_metric('mrr')
    hits10 = metric_results.get_metric('hits@10')
except Exception:
    metric_dict = metric_results.to_flat_dict()
    mrr = metric_dict.get('both.realistic.inverse_harmonic_mean_rank')
    hits10 = metric_dict.get('both.realistic.hits_at_10')

print('=' * 40)
print('[DONE] TransE training completed successfully')
print(f'[INFO] MRR: {mrr:.4f}')
print(f'[INFO] Hits@10: {hits10:.4f}')
print('=' * 40)

model_dir = BASE_DIR / 'models' / '2024_12' / 'transe-icsa-seed-1'
model_dir.mkdir(parents=True, exist_ok=True)

result_transe.save_to_directory(model_dir)
print(f'[DONE] Saved: {model_dir}')

[INFO] Using device: cuda


Training epochs on cuda:0: 100%|██████████| 300/300 [1:06:10<00:00, 13.24s/epoch, loss=0.0992, prev_loss=0.0993]
Evaluating on cuda:0: 100%|██████████| 10.0k/10.0k [07:35<00:00, 21.9triple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 544.67s seconds


[DONE] TransE training completed successfully
[INFO] MRR: 0.4564
[INFO] Hits@10: 0.6359


INFO:pykeen.triples.triples_factory:Stored TriplesFactory(num_entities=448384, num_relations=20, create_inverse_triples=False, num_triples=1696145) to file:///C:/Workspace/icsa-threat-kg/models/2024_12/transe-icsa-seed-1/training_triples
INFO:pykeen.pipeline.api:Saved to directory: C:\Workspace\icsa-threat-kg\models\2024_12\transe-icsa-seed-1


[DONE] Saved: C:\Workspace\icsa-threat-kg\models\2024_12\transe-icsa-seed-1


In [19]:
import torch

model_path = BASE_DIR / 'models' / '2024_12' / 'transe-icsa-seed-0' / 'trained_model.pkl'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[INFO] Using device: {device}')

if not model_path.exists():
    raise FileNotFoundError(f'[ERROR] Model file not found: {model_path}')

model_transe = torch.load(model_path, map_location=device, weights_only=False)
model_transe = model_transe.to(device)
model_transe.eval()

print('=' * 40)
print('[DONE] TransE model loaded successfully')
print(f'[INFO] Model: {model_transe.__class__.__name__}')
print(f'[INFO] Number of parameters: {sum(p.numel() for p in model_transe.parameters()):,}')
print(f'[INFO] Loaded from: {model_path}')
print('=' * 40)

[INFO] Using device: cuda
[DONE] TransE model loaded successfully
[INFO] Model: TransE
[INFO] Number of parameters: 89,680,800
[INFO] Loaded from: C:\Workspace\icsa-threat-kg\models\2024_12\transe-icsa-seed-0\trained_model.pkl


In [18]:
from pykeen.evaluation import RankBasedEvaluator

print('[INFO] Re-evaluating TransE model on the test set ...')

evaluator = RankBasedEvaluator(
    metrics=['hits@k'],
    metrics_kwargs=[{'k': 20}],
    add_defaults=True
)

evaluation_results = evaluator.evaluate(
    model=model_transe,
    mapped_triples=testing_factory.mapped_triples,
    additional_filter_triples=[
        training_factory.mapped_triples,
        validation_factory.mapped_triples
    ],
    batch_size=2048,
    device=device
)

try:
    mrr = evaluation_results.get_metric('mrr')
    mr = evaluation_results.get_metric('mean_rank')
    hits20 = evaluation_results.get_metric('hits@20')
    hits10 = evaluation_results.get_metric('hits@10')
    hits3 = evaluation_results.get_metric('hits@3')
    hits1 = evaluation_results.get_metric('hits@1')
except Exception:
    metric_dict = evaluation_results.to_flat_dict()
    mrr = metric_dict.get('both.realistic.inverse_harmonic_mean_rank')
    mr = metric_dict.get('both.realistic.arithmetic_mean_rank')
    hits20 = metric_dict.get('both.realistic.hits_at_20')
    hits10 = metric_dict.get('both.realistic.hits_at_10')
    hits3 = metric_dict.get('both.realistic.hits_at_3')
    hits1 = metric_dict.get('both.realistic.hits_at_1')

print('=' * 40)
print('[DONE] TransE evaluation completed successfully')
print(f'[INFO] MRR: {mrr:.3f}')
print(f'[INFO] MR: {int(mr):,}')
print(f'[INFO] Hits@20: {hits20:.3f}')
print(f'[INFO] Hits@10: {hits10:.3f}')
print(f'[INFO] Hits@3: {hits3:.3f}')
print(f'[INFO] Hits@1: {hits1:.3f}')
print('=' * 40)

[INFO] Re-evaluating TransE model on the test set ...


Evaluating on cuda:0:   0%|          | 0.00/10.0k [00:01<?, ?triple/s]


KeyboardInterrupt: 

In [20]:
matching_cve_id = training_factory.relation_to_id['MatchingCVE']
matching_cwe_id = training_factory.relation_to_id['MatchingCWE']

mask_matching_cve = testing_factory.mapped_triples[:, 1] == matching_cve_id
mask_matching_cwe = testing_factory.mapped_triples[:, 1] == matching_cwe_id

test_cpe_cve_triples = testing_factory.mapped_triples[mask_matching_cve]
test_cve_cwe_triples = testing_factory.mapped_triples[mask_matching_cwe]

print('=' * 40)
print('[DONE] Target test triples extracted successfully')
print(f'[INFO] MatchingCVE relation ID: {matching_cve_id}')
print(f'[INFO] MatchingCWE relation ID: {matching_cwe_id}')
print(f'[INFO] Number of CPE-CVE test triples: {len(test_cpe_cve_triples):,}')
print(f'[INFO] Number of CVE-CWE test triples: {len(test_cve_cwe_triples):,}')
print('=' * 40)

[DONE] Target test triples extracted successfully
[INFO] MatchingCVE relation ID: 7
[INFO] MatchingCWE relation ID: 8
[INFO] Number of CPE-CVE test triples: 7,853
[INFO] Number of CVE-CWE test triples: 135


In [21]:
import torch
import numpy as np
from collections import defaultdict


def evaluate_cve_to_cpe(model, test_triples, filter_triples, entities_subset,
                        device, batch_size, slice_size):

    # (1) Move the Model to the Target Device and Set Evaluation Mode
    model = model.to(device)
    model.eval()

    # (2) Convert filter_triples to a Unified NumPy Format
    if isinstance(filter_triples, torch.Tensor):
        filter_triples_np = filter_triples.detach().cpu().numpy()
    else:
        filter_triples_np = np.asarray(filter_triples)

    # (4) Prepare the Candidate Head Entity Subset
    if isinstance(entities_subset, torch.Tensor):
        candidate_heads = entities_subset.to(device)
    else:
        candidate_heads = torch.tensor(entities_subset, dtype=torch.long, device=device)

    if candidate_heads.numel() == 0:
        raise ValueError('[ERROR] entities_subset is empty.')

    # (3) Build a Mapping from (relation, tail) to True Head Entities
    rt_to_true_heads = defaultdict(set)
    for h, r, t in filter_triples_np:
        rt_to_true_heads[(int(r), int(t))].add(int(h))

    candidate_heads_list = candidate_heads.detach().cpu().tolist()
    candidate_head_to_pos = {entity_id: idx for idx, entity_id in enumerate(candidate_heads_list)}

    test_triples = test_triples.to(device)
    ranks = []

    with torch.inference_mode():
        total = len(test_triples)

        # (5) Process the Test Triples in Batches
        for start in range(0, total, batch_size):
            end = min(start + batch_size, total)
            batch = test_triples[start:end]

            # (6) Extract (relation, tail) Pairs from Each Batch
            true_heads = batch[:, 0]
            rels = batch[:, 1]
            tails = batch[:, 2]

            rt_batch = torch.stack([rels, tails], dim=1)

            # (7) Compute Scores for Candidate Head Entities
            subset_scores = model.predict_h(rt_batch, heads=candidate_heads, slice_size=slice_size)

            for i in range(batch.shape[0]):
                h = int(true_heads[i].item())
                r = int(rels[i].item())
                t = int(tails[i].item())

                if h not in candidate_head_to_pos:
                    raise ValueError(f'[ERROR] True head entity ID {h} is not included in entities_subset.')

                scores_i = subset_scores[i].clone()
                true_heads_for_rt = rt_to_true_heads[(r, t)]

                # (8) Filter Out Other True Head Entities
                filter_positions = [
                    candidate_head_to_pos[cand_h]
                    for cand_h in true_heads_for_rt
                    if cand_h != h and cand_h in candidate_head_to_pos
                ]

                if filter_positions:
                    filter_positions = torch.tensor(filter_positions, dtype=torch.long, device=device)
                    scores_i[filter_positions] = -torch.inf

                true_pos = candidate_head_to_pos[h]
                true_score = scores_i[true_pos]

                # (9) Identify the True Head Score and Compute Its Rank
                optimistic_rank = 1 + torch.sum(scores_i > true_score).item()
                pessimistic_rank = torch.sum(scores_i >= true_score).item()
                realistic_rank = 0.5 * (optimistic_rank + pessimistic_rank)

                ranks.append(realistic_rank)

    return np.asarray(ranks, dtype=np.float64)

In [22]:
connected_cpe_file = BASE_DIR / 'data' / 'processed' / 'kg' / '2024_12' / 'connected_cpe_list.txt'

if not connected_cpe_file.exists():
    raise FileNotFoundError(f'[ERROR] Connected CPE list not found: {connected_cpe_file}')

with open(connected_cpe_file, 'r', encoding='utf-8') as f:
    connected_cpe_names = [line.strip() for line in f if line.strip()]

connected_cpe_ids = sorted(
    training_factory.entity_to_id[cpe_name]
    for cpe_name in connected_cpe_names
    if cpe_name in training_factory.entity_to_id
)

missing_connected_cpe = [
    cpe_name
    for cpe_name in connected_cpe_names
    if cpe_name not in training_factory.entity_to_id
]

all_true_triples = torch.cat(
    [
        training_factory.mapped_triples,
        validation_factory.mapped_triples,
        testing_factory.mapped_triples
    ],
    dim=0
)

print('=' * 40)
print('[INFO] Preparing CVE->CPE evaluation ...')
print(f'[INFO] Connected CPE list file: {connected_cpe_file}')
print(f'[INFO] Number of connected CPE names: {len(connected_cpe_names):,}')
print(f'[INFO] Number of connected CPE IDs: {len(connected_cpe_ids):,}')
print(f'[INFO] Number of missing CPE names: {len(missing_connected_cpe):,}')
print(f'[INFO] Number of CPE-CVE test triples: {len(test_cpe_cve_triples):,}')
print(f'[INFO] Number of filter triples: {len(all_true_triples):,}')
print('=' * 40)

ranks_cve_to_cpe = evaluate_cve_to_cpe(model_transe, test_cpe_cve_triples, all_true_triples, connected_cpe_ids,
                                       device='cuda', batch_size=128, slice_size=2048)

mrr = np.mean(1.0 / ranks_cve_to_cpe)
mr = np.mean(ranks_cve_to_cpe)
hits20 = np.mean(ranks_cve_to_cpe <= 20)
hits10 = np.mean(ranks_cve_to_cpe <= 10)
hits3 = np.mean(ranks_cve_to_cpe <= 3)
hits1 = np.mean(ranks_cve_to_cpe <= 1)

print('=' * 40)
print('[DONE] CVE->CPE evaluation completed successfully')
print(f'[INFO] MRR: {mrr:.3f}')
print(f'[INFO] MR: {int(mr):,}')
print(f'[INFO] Hits@20: {hits20:.3f}')
print(f'[INFO] Hits@10: {hits10:.3f}')
print(f'[INFO] Hits@3: {hits3:.3f}')
print(f'[INFO] Hits@1: {hits1:.3f}')
print('=' * 40)

[INFO] Preparing CVE->CPE evaluation ...
[INFO] Connected CPE list file: C:\Workspace\icsa-threat-kg\data\processed\kg\2024_12\connected_cpe_list.txt
[INFO] Number of connected CPE names: 121,760
[INFO] Number of connected CPE IDs: 121,760
[INFO] Number of missing CPE names: 0
[INFO] Number of CPE-CVE test triples: 7,853
[INFO] Number of filter triples: 1,716,145
[DONE] CVE->CPE evaluation completed successfully
[INFO] MRR: 0.492
[INFO] MR: 632
[INFO] Hits@20: 0.746
[INFO] Hits@10: 0.670
[INFO] Hits@3: 0.523
[INFO] Hits@1: 0.407


In [23]:
import torch
import numpy as np
from collections import defaultdict


def evaluate_cve_to_cwe(model, test_triples, filter_triples, entities_subset,
                        device, batch_size, slice_size):

    model = model.to(device)
    model.eval()

    if isinstance(filter_triples, torch.Tensor):
        filter_triples_np = filter_triples.detach().cpu().numpy()
    else:
        filter_triples_np = np.asarray(filter_triples)

    if isinstance(entities_subset, torch.Tensor):
        candidate_tails = entities_subset.to(device)
    else:
        candidate_tails = torch.tensor(entities_subset, dtype=torch.long, device=device)

    if candidate_tails.numel() == 0:
        raise ValueError('[ERROR] entities_subset is empty.')

    hr_to_true_tails = defaultdict(set)
    for h, r, t in filter_triples_np:
        hr_to_true_tails[(int(h), int(r))].add(int(t))

    candidate_tails_list = candidate_tails.detach().cpu().tolist()
    candidate_tail_to_pos = {entity_id: idx for idx, entity_id in enumerate(candidate_tails_list)}

    test_triples = test_triples.to(device)
    ranks = []

    with torch.inference_mode():
        total = len(test_triples)

        for start in range(0, total, batch_size):
            end = min(start + batch_size, total)
            batch = test_triples[start:end]

            heads = batch[:, 0]
            rels = batch[:, 1]
            true_tails = batch[:, 2]

            hr_batch = torch.stack([heads, rels], dim=1)

            subset_scores = model.predict_t(hr_batch, tails=candidate_tails, slice_size=slice_size)

            for i in range(batch.shape[0]):
                h = int(heads[i].item())
                r = int(rels[i].item())
                t = int(true_tails[i].item())

                if t not in candidate_tail_to_pos:
                    raise ValueError(f'[ERROR] True tail entity ID {t} is not included in entities_subset.')

                scores_i = subset_scores[i].clone()
                true_tails_for_hr = hr_to_true_tails[(h, r)]

                filter_positions = [
                    candidate_tail_to_pos[cand_t]
                    for cand_t in true_tails_for_hr
                    if cand_t != t and cand_t in candidate_tail_to_pos
                ]

                if filter_positions:
                    filter_positions = torch.tensor(filter_positions, dtype=torch.long, device=device)
                    scores_i[filter_positions] = -torch.inf

                true_pos = candidate_tail_to_pos[t]
                true_score = scores_i[true_pos]

                optimistic_rank = 1 + torch.sum(scores_i > true_score).item()
                pessimistic_rank = torch.sum(scores_i >= true_score).item()
                realistic_rank = 0.5 * (optimistic_rank + pessimistic_rank)

                ranks.append(realistic_rank)

    return np.asarray(ranks, dtype=np.float64)

In [24]:
cwe_candidate_file = BASE_DIR / 'data' / 'processed' / 'kg' / '2024_12' / 'cwe_candidate_list.txt'

if not cwe_candidate_file.exists():
    raise FileNotFoundError(f'[ERROR] CWE candidate list not found: {cwe_candidate_file}')

with open(cwe_candidate_file, 'r', encoding='utf-8') as f:
    cwe_candidate_list = [line.strip() for line in f if line.strip()]

all_true_triples = torch.cat(
    [
        training_factory.mapped_triples,
        validation_factory.mapped_triples,
        testing_factory.mapped_triples,
    ],
    dim=0,
)

cwe_ids = sorted(
    training_factory.entity_to_id[cwe_name]
    for cwe_name in cwe_candidate_list
    if cwe_name in training_factory.entity_to_id
)

missing_cwe = [
    cwe_name
    for cwe_name in cwe_candidate_list
    if cwe_name not in training_factory.entity_to_id
]

print('=' * 40)
print('[INFO] Preparing CVE->CWE evaluation ...')
print(f'[INFO] CWE candidate list file: {cwe_candidate_file}')
print(f'[INFO] Number of CWE candidate names: {len(cwe_candidate_list):,}')
print(f'[INFO] Number of mapped CWE IDs: {len(cwe_ids):,}')
print(f'[INFO] Number of missing CWE names: {len(missing_cwe):,}')
print(f'[INFO] Number of CVE-CWE test triples: {len(test_cve_cwe_triples):,}')
print(f'[INFO] Number of filter triples: {len(all_true_triples):,}')
print('=' * 40)

ranks_cve_to_cwe = evaluate_cve_to_cwe(model_transe, test_cve_cwe_triples, all_true_triples, cwe_ids,
                                       device='cuda', batch_size=128, slice_size=2048)

mrr = np.mean(1.0 / ranks_cve_to_cwe)
mr = np.mean(ranks_cve_to_cwe)
hits20 = np.mean(ranks_cve_to_cwe <= 20)
hits10 = np.mean(ranks_cve_to_cwe <= 10)
hits3 = np.mean(ranks_cve_to_cwe <= 3)
hits1 = np.mean(ranks_cve_to_cwe <= 1)

print('=' * 40)
print('[DONE] CVE->CWE evaluation completed successfully')
print(f'[INFO] MRR: {mrr:.3f}')
print(f'[INFO] MR: {int(mr):,}')
print(f'[INFO] Hits@20: {hits20:.3f}')
print(f'[INFO] Hits@10: {hits10:.3f}')
print(f'[INFO] Hits@3: {hits3:.3f}')
print(f'[INFO] Hits@1: {hits1:.3f}')
print('=' * 40)

[INFO] Preparing CVE->CWE evaluation ...
[INFO] CWE candidate list file: C:\Workspace\icsa-threat-kg\data\processed\kg\2024_12\cwe_candidate_list.txt
[INFO] Number of CWE candidate names: 944
[INFO] Number of mapped CWE IDs: 944
[INFO] Number of missing CWE names: 0
[INFO] Number of CVE-CWE test triples: 135
[INFO] Number of filter triples: 1,716,145
[DONE] CVE->CWE evaluation completed successfully
[INFO] MRR: 0.327
[INFO] MR: 73
[INFO] Hits@20: 0.607
[INFO] Hits@10: 0.519
[INFO] Hits@3: 0.348
[INFO] Hits@1: 0.230
